In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_path = "/kaggle/input/notebooks/jaafarnejm/mouse-dynamics/Predicted_mouse.csv"
df = pd.read_csv(train_path)
df = df.head(250000).copy()

df = df.sort_values(by=['session_id', 'timestamp'])
df['dt'] = df.groupby('session_id')['timestamp'].diff().fillna(0)
df['dx'] = df.groupby('session_id')['screen_x'].diff().fillna(0)
df['dy'] = df.groupby('session_id')['screen_y'].diff().fillna(0)

df['target_dx'] = df.groupby('session_id')['screen_x'].diff().shift(-1)
df['target_dy'] = df.groupby('session_id')['screen_y'].diff().shift(-1)
df = df.dropna()

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

features = ['screen_x', 'screen_y', 'dx', 'dy', 'dt']
targets = ['target_dx', 'target_dy']

df_raw_coords = df[['screen_x', 'screen_y']].values

df[features] = feature_scaler.fit_transform(df[features])
df[targets] = target_scaler.fit_transform(df[targets])

In [ ]:
SEQ_LENGTH = 10 
X_list, y_list, coord_list = [], [], []

idx = 0
for session, group in df.groupby('session_id'):
    group_features = group[features].values
    group_targets = group[targets].values
    group_raw = group[['screen_x', 'screen_y']].values
    
    for i in range(len(group_features) - SEQ_LENGTH):
        X_list.append(group_features[i : i + SEQ_LENGTH])
        y_list.append(group_targets[i + SEQ_LENGTH - 1])
        coord_list.append(group_raw[i + SEQ_LENGTH - 1])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)
coords = np.array(coord_list, dtype=np.float32)

indices = np.arange(X.shape[0])
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
coords_test = coords[test_idx]

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
class MouseLSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2):
        super(MouseLSTMRegressor, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = MouseLSTMRegressor(input_dim=5, hidden_dim=256, output_dim=2, num_layers=2).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

epochs = 30
train_losses, val_losses = [], []
best_loss = float('inf')
patience, patience_counter = 5, 0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)
    
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            epoch_val_loss += loss.item() * batch_X.size(0)
            
    epoch_val_loss = epoch_val_loss / len(test_loader.dataset)
    val_losses.append(epoch_val_loss)
    
    scheduler.step(epoch_val_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_train_loss:.6f} - Val Loss: {epoch_val_loss:.6f}")
    
    if epoch_val_loss < best_loss:
        best_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_mouse_lstm.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered")
            break

In [ ]:
model.load_state_dict(torch.load("best_mouse_lstm.pt"))
model.eval()

all_preds = []
with torch.no_grad():
    for batch_X, _ in test_loader:
        batch_X = batch_X.to(device)
        preds = model(batch_X).cpu().numpy()
        all_preds.append(preds)

y_pred_scaled = np.vstack(all_preds)

y_pred_deltas = target_scaler.inverse_transform(y_pred_scaled)
y_test_deltas = target_scaler.inverse_transform(y_test)

y_test_real = coords_test + y_test_deltas
y_pred_real = coords_test + y_pred_deltas

r2_x = r2_score(y_test_real[:, 0], y_pred_real[:, 0])
r2_y = r2_score(y_test_real[:, 1], y_pred_real[:, 1])
avg_r2 = (r2_x + r2_y) / 2

print(f"R-Squared (X): {r2_x:.4f}")
print(f"R-Squared (Y): {r2_y:.4f}")
print(f"Average R-Squared: {avg_r2:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(y_test_real[:50, 0], y_test_real[:50, 1], 'bo-', label='True Path', alpha=0.6)
plt.plot(y_pred_real[:50, 0], y_pred_real[:50, 1], 'rx--', label='Predicted Path', alpha=0.8)
plt.title("True vs Predicted Trajectory")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss (MSE)")
plt.plot(val_losses, label="Validation Loss (MSE)")
plt.title("Loss Convergence")
plt.legend()
plt.show()